### Importing the necessay libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import timm
from torch.utils.data import DataLoader
import copy
import torch.nn.functional as F

### Defining HyperParameters


### Data Preparation

In [ ]:
BATCH_SIZE = 64
EPOCHS_TEACHER = 10
EPOCHS_STUDENT = 30
LR = 0.001
NUM_CLASSES = 10
IMG_SIZE = 224
PATCH_SIZE = 16  # Size of the patches (16x16)
EMBED_DIM = 192  # Dimension of the embedding (matches ViT-Tiny)
DEPTH = 12       # Number of Transformer blocks
NUM_HEADS = 3    # Number of Attention heads
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
print(torch.cuda.is_available())
print(torch.version.cuda)

True
11.8


In [ ]:
# --- DATA PREP ---
train_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE), # Added: Resize images to IMG_SIZE
    transforms.RandomCrop(IMG_SIZE, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)), # CIFAR-10 standard normalization
])

test_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])


trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)

trainloader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
testloader = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

### CNN Teacher model and VIT teacher model

In [ ]:
def get_cnn_teacher():
    """
    Loads a pre-trained ResNet18 and modifies the last layer for CIFAR-10.
    """
    # Load ResNet18 pre-trained on ImageNet
    model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)

    # Freeze all parameters in the model (backbone)
    for param in model.parameters():
        param.requires_grad = False

    # Replace the last fully connected layer (fc) with a new one for 10 classes.
    # New layers have requires_grad=True by default.
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, NUM_CLASSES)

    return model.to(DEVICE)

def get_vit_teacher():
    """
    Loads a pre-trained ViT and modifies the head for CIFAR-10.
    """
    # Load a tiny ViT pre-trained on ImageNet using timm
    model = timm.create_model('vit_tiny_patch16_224', pretrained=True)

    # Freeze the backbone
    for param in model.parameters():
        param.requires_grad = False

    # Replace the classification head
    # In timm, the head is often called 'head'.
    model.head = nn.Linear(model.head.in_features, NUM_CLASSES)

    return model.to(DEVICE)

### Diet Student model

Instead of writing the diet model from scratch, lets load the pretrained diet model

In [ ]:
class PatchEmbedding(nn.Module):
    """
    Splits the image into patches and embeds them.
    We use a Conv2d layer with kernel_size=patch_size and stride=patch_size
    to achieve splitting and linear projection in one step.
    """
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=768):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.n_patches = (img_size // patch_size) ** 2

        # This convolution effectively splits the image and projects it
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        # x shape: [Batch, 3, 224, 224]
        x = self.proj(x)  # Shape: [Batch, Embed_Dim, 14, 14]
        x = x.flatten(2)  # Shape: [Batch, Embed_Dim, 196]
        x = x.transpose(1, 2)  # Shape: [Batch, 196, Embed_Dim]
        return x

class Attention(nn.Module):
    """
    Standard Multi-Head Self-Attention.
    """
    def __init__(self, dim, num_heads=8, qkv_bias=False, attn_drop=0., proj_drop=0.):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5

        # Query, Key, Value projections combined
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x):
        B, N, C = x.shape
        # qkv shape: [Batch, N, 3, Heads, Head_Dim]
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

class MLP(nn.Module):
    """
    Feed-Forward Network (Multilayer Perceptron) within the transformer block.
    """
    def __init__(self, in_features, hidden_features=None, out_features=None, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = nn.GELU() # Standard activation for Transformers
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x

class TransformerBlock(nn.Module):
    """
    A single Transformer Encoder Block:
    Input -> LayerNorm -> Attention -> Residual -> LayerNorm -> MLP -> Residual
    """
    def __init__(self, dim, num_heads, mlp_ratio=4., qkv_bias=False, drop=0., attn_drop=0.):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = Attention(dim, num_heads=num_heads, qkv_bias=qkv_bias, attn_drop=attn_drop, proj_drop=drop)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = MLP(in_features=dim, hidden_features=int(dim * mlp_ratio), drop=drop)

    def forward(self, x):
        # Pre-Norm architecture (standard for ViT/DeiT)
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

class DeiTFromScratch(nn.Module):
    """
    The full DeiT Model built from components.
    Key difference from ViT: It has TWO tokens (CLS and DIST).
    """
    def __init__(self, img_size=224, patch_size=16, in_chans=3, num_classes=10,
                 embed_dim=192, depth=12, num_heads=3, mlp_ratio=4, dropout = 0.1):
        super().__init__()

        # 1. Patch Embeddings
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_chans, embed_dim)
        num_patches = self.patch_embed.n_patches

        # 2. Learnable Tokens
        # CLS Token: Used for standard classification
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        # DIST Token: Used for distillation (learning from teacher)
        self.dist_token = nn.Parameter(torch.zeros(1, 1, embed_dim))

        # 3. Position Embeddings (Add 2 to num_patches for cls and dist tokens)
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 2, embed_dim))
        self.pos_drop = nn.Dropout(p=dropout)

        # 4. Transformer Encoder Blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(dim=embed_dim, num_heads=num_heads, mlp_ratio=mlp_ratio)
            for _ in range(depth)
        ])

        # 5. Normalization
        self.norm = nn.LayerNorm(embed_dim)

        # 6. Heads (Two separate heads!)
        self.head_cls = nn.Linear(embed_dim, num_classes)
        self.head_dist = nn.Linear(embed_dim, num_classes)

        # Weight initialization
        nn.init.trunc_normal_(self.pos_embed, std=.02)
        nn.init.trunc_normal_(self.cls_token, std=.02)
        nn.init.trunc_normal_(self.dist_token, std=.02)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    def forward(self, x):
        B = x.shape[0]

        # 1. Embed patches
        x = self.patch_embed(x) # [Batch, 196, Embed_Dim]

        # 2. Append CLS and DIST tokens
        cls_tokens = self.cls_token.expand(B, -1, -1)   # [Batch, 1, Embed_Dim]
        dist_tokens = self.dist_token.expand(B, -1, -1) # [Batch, 1, Embed_Dim]

        # Concatenate: [CLS, DIST, Patches...]
        x = torch.cat((cls_tokens, dist_tokens, x), dim=1) # [Batch, 198, Embed_Dim]

        # 3. Add Position Embeddings
        x = x + self.pos_embed
        x = self.pos_drop(x)

        # 4. Pass through Transformer Blocks
        for block in self.blocks:
            x = block(x)

        x = self.norm(x)

        # 5. Extract output tokens
        # x[:, 0] is the CLS token output
        # x[:, 1] is the DIST token output
        return self.head_cls(x[:, 0]), self.head_dist(x[:, 1])

In [ ]:
def get_deit_student():
    """
    Returns our custom built-from-scratch DeiT model.
    """
    print("Initializing Custom DeiT Model from Scratch...")
    model = DeiTFromScratch(
        img_size=IMG_SIZE,
        patch_size=PATCH_SIZE,
        num_classes=NUM_CLASSES,
        embed_dim=EMBED_DIM,
        depth=DEPTH,
        num_heads=NUM_HEADS
    )
    return model.to(DEVICE)

In [ ]:
# Lets train the teacher head

def train_teacher_head(model, loader, epochs):
    """
    Trains only the active (unfrozen) parameters of the teacher model.
    """
    print("Training Teacher Head...")
    # Using CrossEntropyLoss for standard classification
    criterion = nn.CrossEntropyLoss()
    # Optimizer only updates parameters that require gradients (the head)
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)


    for epoch in range(epochs):
        model.eval()
        # Set ONLY the final fully connected layer to train mode
        if hasattr(model, 'fc'):
            model.fc.train()     # For ResNet
        elif hasattr(model, 'head'):
            model.head.train()   # For timm ViT
        running_loss = 0.0
        for inputs, labels in loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE) # Move data to GPU/CPU

            optimizer.zero_grad()           # Clear previous gradients
            outputs = model(inputs)         # Forward pass
            loss = criterion(outputs, labels) # Calculate loss
            loss.backward()                 # Backward pass (gradients)
            optimizer.step()                # Update weights

            running_loss += loss.item()
        print(f"Teacher Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(loader):.4f}")
    return model



### Loss function

lets implement the loss function for hard distillation

In [ ]:
class HardDistillationLoss(nn.Module):
    """
    Implements Hard Distillation Loss as described in the DeiT paper.
    Loss = (1/2) * CE(cls_token, label) + (1/2) * CE(dist_token, teacher_label)
    """
    def __init__(self, teacher_model):
        super().__init__()
        self.teacher = teacher_model
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, inputs, outputs, labels):
        """
        inputs: Input images
        outputs: Tuple (cls_logits, dist_logits) from the student
        labels: True ground truth labels
        """
        cls_logits, dist_logits = outputs

        # Get teacher predictions
        with torch.no_grad(): # No gradients needed for teacher
            teacher_logits = self.teacher(inputs)
            # Hard Distillation: Use the argmax (hard label) of the teacher
            teacher_labels = torch.argmax(teacher_logits, dim=1)

        # Standard classification loss on the class token
        base_loss = self.criterion(cls_logits, labels)

        # Distillation loss on the distillation token matching teacher's hard label
        dist_loss = self.criterion(dist_logits, teacher_labels)

        # Combine losses (0.5 weight each is standard for hard distillation)
        total_loss = 0.5 * base_loss + 0.5 * dist_loss
        return total_loss

In [ ]:
# lets train the student

def train_student(student, teacher, loader, epochs):
    """
    Trains the DeiT student using the provided teacher.
    """
    print("Training Student with Distillation...")
    # Initialize our custom Hard Distillation Loss
    criterion = HardDistillationLoss(teacher)
    # 2. Advanced Optimizer (AdamW with Weight Decay grouping)
    decay_params = []
    no_decay_params = []
    for name, param in student.named_parameters():
        if not param.requires_grad: continue
        # Don't decay biases, norms, or our special tokens
        if param.ndim < 2 or 'bias' in name or 'norm' in name or 'token' in name or 'pos_embed' in name:
            no_decay_params.append(param)
        else:
            decay_params.append(param)

    optim_groups = [
        {'params': decay_params, 'weight_decay': 0.05},
        {'params': no_decay_params, 'weight_decay': 0.0}
    ]
    optimizer = optim.AdamW(optim_groups, lr=LR)

    # 3. Learning Rate Scheduler
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    student.train() # Set student to train mode
    teacher.eval()  # Set teacher to eval mode (important for BatchNorm/Dropout)

    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()

            # Student forward pass. Returns (cls_logits, dist_logits)
            outputs = student(inputs)

            # Calculate distillation loss
            loss = criterion(inputs, outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()
        scheduler.step()
        print(f"Student Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(loader):.4f}")
    return student

In [ ]:
def evaluate_student(model, loader, mode='cls'):
    """
    Evaluates the student model based on the specific prediction mode.
    Modes:
    1. 'cls': Use classification token only.
    2. 'dist': Use distillation token only.
    3. 'avg': Use average of both tokens.
    """
    model.eval() # Set to evaluation mode
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)

            # Get raw logits for both tokens
            cls_logits, dist_logits = model(inputs)

            if mode == 'cls':
                # Case 1: Prediction based purely on class token
                outputs = cls_logits
            elif mode == 'dist':
                # Case 2: Prediction based purely on distillation token
                outputs = dist_logits
            elif mode == 'avg':
                # Case 3: Prediction based on average of both
                outputs = (cls_logits + dist_logits) / 2

            # Get the predicted class (index of max logit)
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Accuracy ({mode} token): {accuracy:.2f}%")

In [ ]:
def run_experiment(teacher_type):
    print(f"\n--- Starting Experiment with Teacher: {teacher_type} ---")

    # 1. Initialize Teacher
    if teacher_type == 'CNN':
        teacher = get_cnn_teacher()
    else:
        teacher = get_vit_teacher()

    # 2. Train Teacher's Head (as per requirement to train the last layer)
    teacher = train_teacher_head(teacher, trainloader, EPOCHS_TEACHER)

    # 3. Initialize Student (Fresh DeiT model)
    student = get_deit_student()

    # 4. Train Student using Distillation
    student = train_student(student, teacher, trainloader, EPOCHS_STUDENT)

    # 5. Evaluate the 3 Cases
    print(f"--- Evaluation Results (Teacher: {teacher_type}) ---")

    # Case 1: Use classification token alone
    evaluate_student(student, testloader, mode='cls')

    # Case 2: Use distillation token alone
    evaluate_student(student, testloader, mode='dist')

    # Case 3: Use combination
    evaluate_student(student, testloader, mode='avg')



In [ ]:
 # Experiment 1: Using CNN Teacher
 run_experiment(teacher_type='CNN')


--- Starting Experiment with Teacher: CNN ---
Training Teacher Head...
Teacher Epoch 1/10, Loss: 0.5811
Teacher Epoch 2/10, Loss: 0.4056
Teacher Epoch 3/10, Loss: 0.3792
Teacher Epoch 4/10, Loss: 0.3677
Teacher Epoch 5/10, Loss: 0.3604
Teacher Epoch 6/10, Loss: 0.3579
Teacher Epoch 7/10, Loss: 0.3530
Teacher Epoch 8/10, Loss: 0.3505
Teacher Epoch 9/10, Loss: 0.3457
Teacher Epoch 10/10, Loss: 0.3452
Initializing Custom DeiT Model from Scratch...
Training Student with Distillation...
Student Epoch 1/30, Loss: 1.8429
Student Epoch 2/30, Loss: 1.5831
Student Epoch 3/30, Loss: 1.4500
Student Epoch 4/30, Loss: 1.3647
Student Epoch 5/30, Loss: 1.2955
Student Epoch 6/30, Loss: 1.2470
Student Epoch 7/30, Loss: 1.1990
Student Epoch 8/30, Loss: 1.1590
Student Epoch 9/30, Loss: 1.1089
Student Epoch 10/30, Loss: 1.0704
Student Epoch 11/30, Loss: 1.0268
Student Epoch 12/30, Loss: 0.9848
Student Epoch 13/30, Loss: 0.9538
Student Epoch 14/30, Loss: 0.9137
Student Epoch 15/30, Loss: 0.8739
Student Epo

In [ ]:
  # Experiment 2: Using Vision Transformer Teacher
run_experiment(teacher_type='ViT')



--- Starting Experiment with Teacher: ViT ---
Training Teacher Head...
Teacher Epoch 1/10, Loss: 0.9334
Teacher Epoch 2/10, Loss: 0.7425
Teacher Epoch 3/10, Loss: 0.7215
Teacher Epoch 4/10, Loss: 0.7177
Teacher Epoch 5/10, Loss: 0.7147
Teacher Epoch 6/10, Loss: 0.7150
Teacher Epoch 7/10, Loss: 0.7112
Teacher Epoch 8/10, Loss: 0.7122
Teacher Epoch 9/10, Loss: 0.7127
Teacher Epoch 10/10, Loss: 0.7094
Initializing Custom DeiT Model from Scratch...
Training Student with Distillation...
Student Epoch 1/30, Loss: 1.8422
Student Epoch 2/30, Loss: 1.5752
Student Epoch 3/30, Loss: 1.4445
Student Epoch 4/30, Loss: 1.3505
Student Epoch 5/30, Loss: 1.2899
Student Epoch 6/30, Loss: 1.2395
Student Epoch 7/30, Loss: 1.1906
Student Epoch 8/30, Loss: 1.1506
Student Epoch 9/30, Loss: 1.1099
Student Epoch 10/30, Loss: 1.0745
Student Epoch 11/30, Loss: 1.0317
Student Epoch 12/30, Loss: 0.9896
Student Epoch 13/30, Loss: 0.9435
Student Epoch 14/30, Loss: 0.9083
Student Epoch 15/30, Loss: 0.8662
Student Epo